In [ ]:
# %matplotlib ipympl

In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
import scienceplots
import matplotlib as mpl

plt.style.use(['science','ieee'])
mpl.rcParams['path.simplify'] = True
mpl.rcParams['path.simplify_threshold'] = 1.0

In [ ]:
plt.get_backend()

In [ ]:
from src.base.lit_module.components.metrics_standard import MyMultiClassAccuracy, MyLoss
from src.base.lit_module.components.metric_gch import GCH_Metric, RGCH_Metric
from src.class_imbalance_ratios.analysis.summarise_metrics import MetricMetadata

In [ ]:
import logging

log_file = Path(os.getcwd()) / "src/log_analysis.log"
logging.basicConfig(filename=log_file, level=logging.DEBUG)

In [ ]:
from src.class_imbalance_ratios.analysis.metric_run_dir_analysis import agg_save_metrics
from src.class_imbalance_ratios.analysis.needed_paths import cls_imb_outputs_dir
from src.class_imbalance_ratios.analysis.needed_names import corr_store_names

In [ ]:
def convert_pt_class_counts_to_parquet(path: Path, save_path: Path) -> None: 
    class_counts = torch.load(path)
    assert type(class_counts) == dict, f"Class counts is not type dict. Got {type(class_counts)} instead."
    class_counts = {str(k) : [v] for k,v in class_counts.items()} # dict must be in form [str, array], int as key, scalar as value are not accepted
    data_df = pd.DataFrame(class_counts)
    data_df.to_parquet(save_path)      

def delete_all_non_extension(outputs_dir: Path) -> None:
    pt_file_paths = sorted(outputs_dir.rglob("*"))
    for pt_path in pt_file_paths: 
        if (not pt_path.suffix) and (os.path.isfile(pt_path)): # do not try .is_file() of Pathlib
            display(pt_path)
            os.remove(pt_path)

def convert_all_pt_metrics_to_parquet(outputs_dir: Path):
    pt_file_paths = sorted(outputs_dir.rglob("*.pt"))
    existing_parquets = sorted(outputs_dir.rglob("*.parquet"))

    for pt_path in pt_file_paths:
        if len(pt_file_paths) == len(existing_parquets):
            display(f"Lengths of pt to parquet files are equal: {len(pt_file_paths)} == {len(existing_parquets)}.\n Aborting conversion...")
            break
        else:
            display(f"Lengths of pt to parquet files are unequal: {len(pt_file_paths)} == {len(existing_parquets)}.\n Proceeding with conversion...")

        if Path(pt_path.parent / Path(pt_path.stem + ".parquet")).exists():
            display("File already exists at: {}".format(pt_path))
            continue
        else: 
            display("Matching parquet file does not exist at: {}. Converting...".format(pt_path))

        save_path = pt_path.parent / Path(pt_path.stem + ".parquet")
        metric = torch.load(pt_path.resolve(), map_location="cpu", weights_only=True, mmap=True)

        data_cols = [k for k in list(metric.keys()) if k != "epochs" and k != "steps"] # Do not add epochs, steps back into classes, they get merged into expected keys, and their state is reloaded when seen as part of labels, as key names, then again as metadata frmo load state dict method
        if "class_counts" in str(pt_path):
            convert_pt_class_counts_to_parquet(pt_path, save_path)
            continue # Continue after handling this file caught in regex

        # Handle metrics discovered
        if "acc" in str(pt_path):
            base_metric = MyMultiClassAccuracy(data_cols) 
        elif "loss" in str(pt_path):
            base_metric = MyLoss()
        elif "rgc" in str(pt_path).lower():
            base_metric = RGCH_Metric(data_cols)
        else:
            base_metric = GCH_Metric(data_cols)

        base_metric.load_state_dict(metric)

        computed = base_metric.compute()

        data_df = pd.DataFrame(computed).replace(float('nan'), np.nan)
        data_df = data_df.replace(100, np.nan)
        data_df = data_df.map(lambda value: value.item() if isinstance(value, torch.Tensor) else value)

        # Convert tensors not addressed by `.compute()`
        clean_epochs = np.asarray(base_metric.metric_state["epochs"]) 
        clean_steps = np.asarray(base_metric.metric_state["steps"]) 

        metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
        multiindex = pd.MultiIndex.from_arrays(
            [
                list(metadata["epochs"]), 
                list(metadata["steps"]) # list of lists format required
            ],
            names=["epochs", "steps"]
        )
        data_df.index = multiindex

        data_df.to_parquet(save_path)   # by default pandas does not append .parquet   

In [ ]:
display("=========== DELETING ALL PARQUET FILES EXPECTED BUT MISSING .PARQUET ===========")
delete_all_non_extension(outputs_dir=cls_imb_outputs_dir)

In [ ]:
display("=========== CONVERTING ALL .PT files to .PARQUET ===========")
convert_all_pt_metrics_to_parquet(outputs_dir=cls_imb_outputs_dir) # Took 30m 50.7s on CIF10 cls_imb locally

In [ ]:
display("=========== DISPLAYING GRADIENT CORRELATION METRICS ===========")
coupled_metric_paths = [
    dict(paths= f"run*/train_{corr_store_names["normed"]}.parquet", prefix="Normalised-GC", summary_mode_extras=["none", "cols"]),
    dict(paths = f"run*/train_{corr_store_names["raw"]}.parquet", prefix="Unnormalised-GC", summary_mode_extras=["none", "cols"]),
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=1, 
    corr_store_names=corr_store_names, 
    run_paths_info=coupled_metric_paths
)

In [ ]:
plt.close('all')

In [ ]:
display("=========== DISPLAYING ACCURACY METRICS ===========")
coupled_metric_paths = [
    dict(paths= f"run*/train_acc.parquet", prefix="Train Accuracy"),
    dict(paths = f"run*/val_acc.parquet", prefix="Validation Accuracy"), # TODO: When visualizing you can superimpose similar aggregates to one plot
    dict(summary_mode_extras=["none", "both", "epochs", "cols"])  # TODO  We may consider decoupling atypical measures that aren't traiditional aggregation across runs e.g. subgrouping - recommended
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=1, 
    corr_store_names=corr_store_names, 
    run_paths_info=coupled_metric_paths
)

In [ ]:
plt.close('all')

In [ ]:
display("=========== DISPLAYING LOSS METRICS ===========")
coupled_metric_paths = [
    dict(paths= f"run*/train_loss.parquet", prefix="Train Accuracy"),
    dict(paths = f"run*/val_loss.parquet", prefix="Validation Accuracy"), # TODO: When visualizing you can superimpose similar aggregates to one plot
    dict(summary_mode_extras=["none", "both", "epochs", "cols"])
]
agg_save_metrics(
    outputs_dir=cls_imb_outputs_dir, 
    version_dir="multijob/R*", 
    limit=1, 
    corr_store_names=corr_store_names, 
    run_paths_info=coupled_metric_paths
)

In [ ]:
plt.close('all')